# Патчи к `benchmark-task-agentic-github-v2bf642(1).ipynb` — issue #16

Автор: агент `anthropic/claude-opus-5@default`, ветка `anthropic-claude-opus-5-default-issue-16`.
Оригинальные ноутбуки не изменены — это отдельный файл с ячейками-заменами.

Ячейки ниже добавляются **после** ячейки со слоем `gh_*` (раздел 2 исходного
ноутбука) и **до** регистрации инструментов агента. Они закрывают 4 пункта issue:

| # | Проблема | Решение в этом ноутбуке |
|---|---|---|
| 1 | Рабочая ветка сбрасывается между раундами (лишний коммит `911c83b`) | `AgentContext` с состоянием на диске (`kbench_agent_state.json`) + guard по slug |
| 2 | `list_files/search_code/read_file` читают `main`, а не рабочую ветку | `build_read_tools(ctx)` — дефолт `from_my_branch=True`, в ответе печатается имя ветки |
| 3 | `MAX_TOOL_OUTPUT` режет ~50 КБ ноутбук | `gh_read_cursor` (курсор `next_start_line`) + `nb_outline` / `nb_cell` / `nb_replace_cell` |
| 4 | Нет исполнения кода и создания issues | `run_python`, `run_repo_file`, `create_issue` |

Требования к токену прежние: `Contents: RW`, `Issues: RW`, `Metadata: R`.

## Fix 1. Рабочая ветка живёт между раундами

In [ ]:
# Проблема: выбранная через create_issue_branch(N) ветка хранилась в переменной
# одного прогона. В новом раунде объект пересоздавался, и первый write_file уходил
# в дефолтную ветку slug (лишний коммит 911c83b в anthropic-claude-opus-5-default).
# Решение: контекст агента с состоянием на диске + жёсткий guard по префиксу slug.
import json
import os

AGENT_STATE_PATH = os.environ.get('KBENCH_STATE', '/kaggle/working/kbench_agent_state.json')


def _state_load():
    try:
        with open(AGENT_STATE_PATH, encoding='utf-8') as f:
            return json.load(f)
    except Exception:
        return {}


def _state_save(state):
    try:
        with open(AGENT_STATE_PATH, 'w', encoding='utf-8') as f:
            json.dump(state, f, ensure_ascii=False)
    except Exception as e:
        print('state save failed:', e)


class AgentContext:
    '''Контекст одного агента: slug модели + текущая рабочая ветка.

    Ветка пишется и в память, и на диск, поэтому переживает границу раунда
    и рестарт процесса. Записать можно только в ветку с префиксом slug.
    '''

    def __init__(self, slug, base=None):
        self.slug = slug
        self.base = base or GITHUB_BASE_BRANCH
        self.work_branch = _state_load().get(slug, {}).get('work_branch') or slug

    def set_branch(self, branch):
        if not branch.startswith(self.slug):
            raise ValueError('guard: ветка ' + branch + ' вне slug ' + self.slug)
        self.work_branch = branch
        st = _state_load()
        st[self.slug] = {'work_branch': branch}
        _state_save(st)
        return branch

    def resolve(self, from_my_branch=True, branch=None):
        '''Единая точка выбора ветки для всех инструментов.'''
        if branch:
            return branch
        return self.work_branch if from_my_branch else self.base

    def round_header(self, round_no, rounds_total):
        '''Строка, которую раннер обязан добавлять в промпт каждого раунда.'''
        return ('Раунд ' + str(round_no) + '/' + str(rounds_total) +
                '. Текущая рабочая ветка: ' + self.work_branch +
                ' (все записи идут туда, main read-only).')


def make_branch_tools(ctx):
    def create_issue_branch(issue_number: int) -> str:
        '''Создаёт (идемпотентно) ветку <slug>-issue-<N> и делает её текущей на все раунды.'''
        name = ctx.slug + '-issue-' + str(int(issue_number))
        msg = gh_create_branch(name, ctx.base)
        ctx.set_branch(name)
        return msg + ' Текущая рабочая ветка: ' + name

    def use_branch(branch: str) -> str:
        '''Явно переключить рабочую ветку (только внутри своего slug).'''
        try:
            ctx.set_branch(branch)
        except ValueError as e:
            return 'ERROR: ' + str(e)
        return 'Рабочая ветка: ' + ctx.work_branch

    def current_branch() -> str:
        '''Показать текущую рабочую ветку (дешёвая проверка в начале раунда).'''
        return ctx.work_branch

    return create_issue_branch, use_branch, current_branch

## Fix 2. Чтение по умолчанию из рабочей ветки

In [ ]:
# Проблема: list_files/search_code/read_file молча читали main, поэтому агент
# не видел собственных коммитов и мог перезаписать свою же работу.
# Решение: дефолт from_my_branch=True + имя ветки в первой строке ответа.

def build_read_tools(ctx):
    def list_files(subdir: str = '', from_my_branch: bool = True) -> str:
        '''Листинг файлов. По умолчанию — текущая рабочая ветка агента.'''
        br = ctx.resolve(from_my_branch)
        return '[branch: ' + br + ']\n' + gh_list_files(br, subdir)

    def search_code(query: str, subdir: str = '', from_my_branch: bool = True) -> str:
        '''grep по дереву ветки (по умолчанию — рабочей). Возвращает path:line: текст.'''
        br = ctx.resolve(from_my_branch)
        return '[branch: ' + br + ']\n' + gh_search_code(query, br, subdir)

    def read_file(path: str, start_line: int = 1, max_lines: int = 0,
                  from_my_branch: bool = True) -> str:
        '''Чтение файла из рабочей ветки; from_my_branch=False — из base (main).'''
        br = ctx.resolve(from_my_branch)
        return '[branch: ' + br + ']\n' + gh_read_cursor(path, br, start_line, max_lines or None)

    def diff_vs_base() -> str:
        '''Что рабочая ветка меняет относительно base.'''
        d = gh_compare(ctx.base, ctx.work_branch)
        if not d:
            return 'ERROR: сравнение недоступно'
        return ('ahead_by=' + str(d['ahead_by']) + ', behind_by=' + str(d['behind_by']) +
                '\nfiles: ' + ', '.join(d['files']) +
                '\ncommits: ' + ' | '.join(d['commits']))

    return list_files, search_code, read_file, diff_vs_base

## Fix 3. Большие файлы и ноутбуки: курсор вместо слепой обрезки

In [ ]:
# Проблема: _clip резал ответ на MAX_TOOL_OUTPUT без указания, где остановился;
# минифицированный ipynb (одна строка на ~49 КБ) нельзя было прочитать вообще.
# Решение: страничное чтение с курсором + инструменты, понимающие структуру ipynb.

def _gh_text(path, branch):
    '''Сырой текст файла ветки, без обрезки.'''
    resp = _gh_request('GET', REPO_PATH + '/contents/' + path, params={'ref': branch})
    if resp.status_code != 200:
        return None
    data = resp.json()
    if data.get('encoding') != 'base64':
        return None
    return base64.b64decode(data['content']).decode('utf-8', errors='replace')


def gh_read_cursor(path, branch, start_line=1, max_lines=None):
    '''Читает срез файла и всегда сообщает, где остановился.

    Хвост ответа: ...[cursor] next_start_line=N из M — агент продолжает чтение
    следующим вызовом, а не теряет остаток файла.
    Отдельно обрабатывается случай одной сверхдлинной строки (минифицированный JSON):
    она отдаётся кусками по символам.
    '''
    text = _gh_text(path, branch)
    if text is None:
        return 'ERROR: ' + path + ' недоступен в ветке ' + branch
    lines = text.splitlines()
    total = len(lines)
    if total == 1 and len(text) > MAX_TOOL_OUTPUT:
        off = max(0, int(start_line) - 1)
        piece = text[off:off + MAX_TOOL_OUTPUT - 200]
        nxt = off + len(piece) + 1
        tail = ('\n...[cursor] one-line file, next_start_line=' + str(nxt) +
                ' (позиция в символах) из ' + str(len(text)))
        return piece + (tail if nxt <= len(text) else '\n...[eof]')
    budget = MAX_TOOL_OUTPUT - 200
    start = max(1, int(start_line))
    limit = total if not max_lines else min(total, start - 1 + int(max_lines))
    out, used, i = [], 0, start - 1
    while i < limit and used + len(lines[i]) + 1 <= budget:
        out.append(lines[i])
        used += len(lines[i]) + 1
        i += 1
    if i == start - 1 and i < limit:
        out.append(lines[i][:budget])
        i += 1
    if i < total:
        tail = '\n...[cursor] next_start_line=' + str(i + 1) + ' из ' + str(total)
    else:
        tail = '\n...[eof] всего строк ' + str(total)
    return '\n'.join(out) + tail


def _cell_src(cell):
    src = cell.get('source')
    return ''.join(src) if isinstance(src, list) else (src or '')


def build_nb_tools(ctx):
    def nb_outline(path: str, from_my_branch: bool = True) -> str:
        '''Оглавление ipynb: индекс, тип, размер и первая строка каждой ячейки.

        Дешёвая карта 50 КБ ноутбука вместо чтения его целиком.
        '''
        br = ctx.resolve(from_my_branch)
        text = _gh_text(path, br)
        if text is None:
            return 'ERROR: ' + path + ' недоступен в ветке ' + br
        try:
            nb = json.loads(text)
        except ValueError as e:
            return 'ERROR: не JSON: ' + str(e)
        rows = []
        for i, c in enumerate(nb.get('cells', [])):
            s = _cell_src(c)
            head = (s.strip().splitlines() or [''])[0][:100]
            rows.append(str(i) + ' [' + c.get('cell_type', '?') + '] ' +
                        str(len(s)) + ' B | ' + head)
        return _clip('cells: ' + str(len(rows)) + '\n' + '\n'.join(rows))

    def nb_cell(path: str, index: int, from_my_branch: bool = True) -> str:
        '''Исходник одной ячейки ipynb по индексу.'''
        br = ctx.resolve(from_my_branch)
        text = _gh_text(path, br)
        if text is None:
            return 'ERROR: ' + path + ' недоступен в ветке ' + br
        cells = json.loads(text).get('cells', [])
        i = int(index)
        if i < 0 or i >= len(cells):
            return 'ERROR: индекс вне диапазона 0..' + str(len(cells) - 1)
        return _clip(_cell_src(cells[i]))

    def nb_replace_cell(path: str, index: int, new_source: str, message: str = '') -> str:
        '''Точечно заменяет исходник одной ячейки: не надо перезаписывать весь JSON.'''
        br = ctx.work_branch
        text = _gh_text(path, br)
        if text is None:
            return 'ERROR: ' + path + ' недоступен в ветке ' + br
        nb = json.loads(text)
        cells = nb.get('cells', [])
        i = int(index)
        if i < 0 or i >= len(cells):
            return 'ERROR: индекс вне диапазона 0..' + str(len(cells) - 1)
        cells[i]['source'] = new_source
        if cells[i].get('cell_type') == 'code':
            cells[i]['outputs'] = []
            cells[i]['execution_count'] = None
        dumped = json.dumps(nb, ensure_ascii=False, indent=1)
        return gh_write_file(path, dumped, br,
                            message or (path + ': заменена ячейка ' + str(i)))

    return nb_outline, nb_cell, nb_replace_cell

## Fix 4. Исполнение кода и создание issues

In [ ]:
# Проблема: агент не мог ни запустить свой код (писал вслепую), ни завести issue.
# Решение: изолированный запуск в отдельном процессе с таймаутом + POST /issues.
import subprocess
import sys
import tempfile
import textwrap

RUN_TIMEOUT_DEFAULT = 30


def _run_file(dirpath, filename, argv, timeout_sec):
    try:
        p = subprocess.run(
            [sys.executable, filename] + argv,
            capture_output=True, text=True, timeout=timeout_sec, cwd=dirpath,
            env={'PATH': os.environ.get('PATH', ''), 'HOME': dirpath,
                 'PYTHONDONTWRITEBYTECODE': '1'},
        )
    except subprocess.TimeoutExpired:
        return 'ERROR: превышен таймаут ' + str(timeout_sec) + ' c'
    return _clip('exit=' + str(p.returncode) +
                 '\n--- stdout ---\n' + p.stdout +
                 '\n--- stderr ---\n' + p.stderr)


def run_python(code: str, timeout_sec: int = RUN_TIMEOUT_DEFAULT) -> str:
    '''Выполняет Python-код в отдельном процессе и возвращает exit code, stdout, stderr.

    Окружение урезано (нет GITHUB_TOKEN в env), рабочая директория временная,
    есть таймаут — чтобы агент проверял свои правки, а не гадал.
    '''
    with tempfile.TemporaryDirectory() as d:
        with open(os.path.join(d, 'snippet.py'), 'w', encoding='utf-8') as fh:
            fh.write(textwrap.dedent(code))
        return _run_file(d, 'snippet.py', [], timeout_sec)


def make_run_repo_file(ctx):
    def run_repo_file(path: str, args: str = '', timeout_sec: int = 60,
                      from_my_branch: bool = True) -> str:
        '''Скачивает файл из ветки во временный каталог и запускает его (например, тесты).'''
        br = ctx.resolve(from_my_branch)
        text = _gh_text(path, br)
        if text is None:
            return 'ERROR: ' + path + ' недоступен в ветке ' + br
        with tempfile.TemporaryDirectory() as d:
            name = os.path.basename(path)
            with open(os.path.join(d, name), 'w', encoding='utf-8') as fh:
                fh.write(text)
            return _run_file(d, name, args.split() if args else [], timeout_sec)

    return run_repo_file


def create_issue(title: str, body: str = '', labels: str = '') -> str:
    '''Создаёт новый issue в репозитории (нужен scope Issues: Read and write).'''
    payload = {'title': title, 'body': body}
    if labels:
        payload['labels'] = [x.strip() for x in labels.split(',') if x.strip()]
    resp = _gh_request('POST', REPO_PATH + '/issues', json=payload)
    if resp.status_code != 201:
        return _err('create_issue', resp) + '  (нужен scope Issues: RW)'
    return 'Issue создан: ' + resp.json()['html_url']

## Сборка набора инструментов и что поменять в раннере

In [ ]:
import functools


def build_toolset(slug, base=None):
    '''Возвращает (ctx, tools) для одной модели. ctx создаётся ОДИН раз и
    переиспользуется во всех раундах — именно это чинит пункт 1 issue #16.'''
    ctx = AgentContext(slug, base)
    create_issue_branch, use_branch, current_branch = make_branch_tools(ctx)
    list_files, search_code, read_file, diff_vs_base = build_read_tools(ctx)
    nb_outline, nb_cell, nb_replace_cell = build_nb_tools(ctx)
    run_repo_file = make_run_repo_file(ctx)

    def write_file(path: str, content: str, commit_message: str = '') -> str:
        '''Полная запись файла в текущую рабочую ветку.'''
        return gh_write_file(path, content, ctx.work_branch, commit_message or None)

    def patch_file(path: str, old_text: str, new_text: str) -> str:
        '''Замена первого вхождения old_text на new_text в файле рабочей ветки.'''
        text = _gh_text(path, ctx.work_branch)
        if text is None:
            return 'ERROR: ' + path + ' недоступен в ветке ' + ctx.work_branch
        if old_text not in text:
            return 'ERROR: фрагмент не найден'
        return gh_write_file(path, text.replace(old_text, new_text, 1), ctx.work_branch,
                             path + ': patch')

    tools = [
        current_branch, create_issue_branch, use_branch,
        list_files, search_code, read_file, diff_vs_base,
        nb_outline, nb_cell, nb_replace_cell,
        write_file, patch_file,
        run_python, run_repo_file, create_issue,
    ]
    return ctx, tools


# Чек-лист изменений в раннере (движок раундов):
# 1. ctx/tools строятся один раз на модель, а не на раунд;
# 2. в системный промпт каждого раунда добавляется ctx.round_header(i, n)
#    -> модель видит текущую ветку и не пишет в дефолтную;
# 3. состояние дублируется в kbench_agent_state.json (переживает рестарт ядра);
# 4. функции-инструменты регистрируются с functools.wraps, иначе схема
#    вырождается в {args, kwargs} (известный баг проекта);
# 5. критерий github_issue_resolution дополняется проверкой,
#    что коммиты попали именно в <slug>-issue-<N>, а не в <slug>.